# 단일 결정 트리(Decision Tree) — 붓꽃 품종 · 당뇨병 여부 분류

수업 중 작성한 필기 `붓꽃데이터EDA` · `결정트리분류실습` · `결정트리분류실습02` · `SVM_분류실습`(뒷부분) 과 개념 메모 `notbook04.py` 의 트리 부분을 **실행 가능하도록 하나로 정리한 노트**입니다.
(교재 `머신러닝` p.66~81 — 주요 분류 모델: 단일 결정 트리)

필기는 당뇨병 데이터를 **불러오기까지만** 되어 있어서, 교재 p.79~81 순서대로 학습 · 평가까지 채웠습니다.
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `iris_dataset.csv` · `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> `scikit-learn` · `matplotlib` · `plotly` 가 필요합니다. SVM 은 [`svm.ipynb`](./svm.ipynb) 에 따로 정리했습니다.

## 1. 단일 결정 트리란? (교재 p.66, p.69~70)

**특정 특성의 임계값(Threshold)을 기준으로 데이터를 축에 수직인 경계로 반복 분할해서, 영역 안의 불순도(Impurity)를 최소화하는 대표적인 비선형 알고리즘.**

"데이터 분할의 논리" = **IF-THEN 규칙** (AI 가 하는 스무고개)

```
꽃잎 너비 <= 0.8 ?
├─ 예  → setosa                          (불순도 0, 끝)
└─ 아니오 → 꽃잎 너비 <= 1.75 ?
           ├─ 예  → 꽃잎 길이 <= 4.95 ? → … 대부분 versicolor
           └─ 아니오 → … 대부분 virginica
```

- 질문을 거듭할수록 그룹의 **불순도가 낮아지며** 한 가지 품종으로 나뉜다.
- 이름이 **결정 트리**인 이유 — 일련의 분류 규칙 적용(= **의사결정**)으로 데이터를 순차적으로 분할하는 과정을 **나무 모양**으로 그린 것이기 때문
- **해석 가능성 (Explainability)** — "AI 가 그냥 setosa 라고 했어요" 가 아니라 "**꽃잎 너비가 0.8cm 이하라서** setosa 입니다" 라고 근거를 설명할 수 있다.

| 장점 | 주의 |
|------|------|
| 결과를 **그림으로 보고 설명**할 수 있다 | 끝까지 나누면 학습용에 **과대적합**되기 쉽다 → `max_depth` 로 깊이 제한 |
| **표준화가 필요 없다** — 한 특성의 임계값으로만 자르므로 단위 · 범위가 상관없다 | 축에 수직인 경계만 그어서 대각선 경계는 계단 모양으로 흉내 낸다 |

## 2. 불순도(Impurity) — 지니 vs 엔트로피 (교재 p.66~68)

**서로 다른 데이터가 섞여 있는 정도.** 다양한 클래스가 섞일수록 높다.

| 상태 | 불순도 |
|------|--------|
| 한 그룹에 두 클래스가 **정확히 반반** — 어느 쪽인지 도저히 예측할 수 없음 | **최악** (지니 0.5, 엔트로피 1.0) |
| 한 영역에 A 클래스만 완벽하게 모여 있음 (**순도 100%**) | **0** |

결정 트리의 목표 = 뒤죽박죽 섞인 데이터(불순도 ↑)를 **끼리끼리 모인 상태(불순도 ↓)** 로 나누는 것.

| 지표 | 공식 | 특징 |
|------|------|------|
| **지니 불순도** (Gini) | $G_i = 1 - \sum_{k=1}^{n} p_{i,k}^2$ | **기본값** `criterion='gini'`, 계산이 빠르다 |
| **엔트로피** (Entropy) | $H_i = -\sum_{k=1,\,p_{i,k} \ne 0}^{n} p_{i,k} \log_2 p_{i,k}$ | 정보 이론의 혼잡도 개념, `criterion='entropy'` |

($p_{i,k}$ = i 번째 노드의 학습 샘플 중 클래스 k 의 비율)

In [ ]:
import numpy as np

# 지니 불순도 / 엔트로피를 직접 계산하는 함수
def gini(counts):
    p = np.array(counts) / sum(counts)       # 클래스별 비율
    return 1 - np.sum(p ** 2)

def entropy(counts):
    p = np.array(counts) / sum(counts)
    p = p[p > 0]                             # 비율 0 인 클래스는 빼고 계산 (log 0 방지)
    return -np.sum(p * np.log2(p)) + 0.0     # + 0.0 : -0.0 을 0.0 으로 표시

# [클래스0 개수, 클래스1 개수, ...]
examples = {
    '한 클래스만 (순도 100%)': [40, 0],
    '9 : 1 로 섞임': [36, 4],
    '반반 (최악, 2클래스)': [20, 20],
    '붓꽃 루트 노드 [40, 40, 40]': [40, 40, 40],
}
for name, counts in examples.items():
    print(f"{name:28s} → 지니 {gini(counts):.3f} / 엔트로피 {entropy(counts):.3f}")

> - 붓꽃 3품종이 40개씩 섞인 루트 노드의 지니는 **0.667** — 아래 4단원에서 그린 트리의 맨 위 상자 `gini = 0.667` 과 같다.
> - 2클래스 반반이면 지니 0.5 · 엔트로피 1.0 으로 **최대**다.

> **📝 고친 부분** (필기 주석)
> - "최악 불순도 : 50:50" → 맞다 (2클래스일 때)
> - "**1:1 보다 1:1:1 나눈 게 불순도 낮아진다**" → 반대다. 같은 비율로 섞인 클래스가 **많을수록 불순도가 더 높다** (1:1 → 지니 0.5, 1:1:1 → 지니 0.667)

## 3. 트리 구성 요소 · 클래스 확률 · CART (교재 p.71~73)

### 트리의 주요 구성 요소

| 이름 | 뜻 |
|------|-----|
| **루트 노드** (Root Node) | 맨 위의 첫 분할 노드 — 전체 학습 데이터에서 시작 |
| **결정 노드** (Decision Node) | 중간에서 다시 분할하는 노드 |
| **리프 노드** (Leaf Node) | 더 이상 나누지 않고 **최종 클래스(예측값)가 결정되는** 맨 아래 노드 |
| **깊이** (depth) | 루트에서 몇 단계 내려왔는가 |

**노드 정보** (`plot_tree` 상자 안의 4줄)

| 줄 | 뜻 |
|----|-----|
| `petal width (cm) <= 0.8` | **Threshold** — 분할 조건. 참이면 **왼쪽(True)**, 거짓이면 **오른쪽(False)** |
| `gini = 0.667` | 이 노드의 **불순도** |
| `samples = 120` | 이 노드에 도착한 **샘플 수** |
| `value = [40, 40, 40]` | **클래스별** 샘플 수 |

### 클래스 확률 추정 (p.72)

샘플이 어떤 **리프 노드 L** 에 도착하면, 그 리프에 있는 **학습 데이터의 클래스 비율**을 확률로 쓴다.

$$P(Y = k \mid X) = \frac{N_{L,k}}{N_L} \qquad (N_L : \text{리프의 샘플 수},\ N_{L,k} : \text{그중 클래스 k 의 수})$$

→ 가장 높은 확률의 클래스를 최종 예측(`predict`)으로 반환한다.

### CART 알고리즘 (p.73)

scikit-learn 이 결정 트리를 학습할 때 쓰는 알고리즘 (Classification And Regression Trees).

- **이진 트리**만 만든다 — 리프를 뺀 모든 노드는 **항상 2개**로 나뉜다
- **분할 기준** — 특성 하나(k)와 임계값($t_k$)을 골라, 두 자식의 불순도가 **가장 낮아지도록** 나눈다
- **정지 기준** — 미리 정한 **최대 깊이(`max_depth`)** 에 도달하거나, 더 이상 불순도를 줄일 분할이 없으면 멈춘다

> **📝 고친 부분** (`notbook04.py` 메모)
> - "**단일 일정 트리**" → 단일 **결정** 트리. "시각화 편하다, 결과물 설명 가능성" → 맞다
> - "결정 트리인 이유 : (의사결정)을 통해 데이터를 순차적으로 분할하는 과정" → 맞다
> - "**1/N 데이터 개수 만큼 자른다**", "분류하는 만큼 나눈다" → 데이터 개수로 자르는 게 아니다. **특성 하나의 임계값**으로 **항상 둘**로 나누고(CART), 리프가 순수해지거나 `max_depth` 에 닿을 때까지 반복한다
> - "루트 노드(**부모**) : 첫 번째 단계 / 리프 노드(**자식**) : 최종 단계" → 부모/자식은 **위아래 관계**를 부르는 말이다. 루트는 부모가 없는 맨 위 노드, 리프는 **자식이 없는** 맨 아래 노드다 (중간 노드는 부모이면서 자식)
> - "노드 정보 : Threshold, gini/entropy, sample, value" → 맞다 (`sample` → `samples`)
> - "부모 노드는 자식 노드의 불순도가 낮아지길 바란다" → 맞다. 그래서 자식 불순도가 가장 낮아지는 분할을 고른다
> - "노드 깊이만큼 나눈다 (불순도 줄어든다)" → 깊어질수록 학습용 불순도는 줄지만, 너무 깊으면 **과대적합** (6단원에서 확인)
> - "클래스 확률 추정 : 결정 트리는 샘플이 어떤 리프 노드에 도달하면," 에서 끊겨 있던 문장 → 위 공식으로 완성
> - "True ← 경계 → False" → `plot_tree` 에서 조건이 **참이면 왼쪽, 거짓이면 오른쪽**

## 4. 붓꽃(iris) 데이터 — EDA (교재 p.74, 필기 `붓꽃데이터EDA`)

**150개**의 붓꽃 데이터, 특성 4개 + 품종 레이블.

| 컬럼 | 뜻 |
|------|-----|
| `sepal length (cm)` | 꽃받침의 길이 |
| `sepal width (cm)` | 꽃받침의 너비 |
| `petal length (cm)` | 꽃잎의 길이 |
| `petal width (cm)` | 꽃잎의 너비 |
| `label` | 품종 (setosa · versicolor · virginica) |

In [ ]:
import pandas as pd
import plotly.express as px

# 데이터 불러오기 (노트북과 같은 폴더)
df = pd.read_csv('./iris_dataset.csv')

print("모양 :", df.shape)
print(f"컬럼별 누락 데이터 수 : \n{df.isnull().sum()}")

print("-" * 40)

# 정답(품종) 분포 → 균형 / 불균형 확인
print(f"품종별 개수 : \n{df['label'].value_counts()}")

In [ ]:
# 컬럼 이름 : 앞의 4개는 특성, 마지막은 정답
names = df.columns
features = names[:-1]
label = names[-1]
print(list(names))

In [ ]:
# 특성별 분포 + 품종별로 겹쳐 그리기 (특성 4개를 반복문으로)
titles = ['꽃받침의 길이', '꽃받침의 너비', '꽃잎의 길이', '꽃잎의 너비']

for col, title in zip(features, titles):
    # 전체 분포
    fig = px.histogram(data_frame=df, x=col, nbins=30, title=f'{title} 분포')
    fig.update_traces(marker_line_color='black', marker_line_width=1)   # 막대 경계선
    fig.show()

    # 품종별 분포 (겹쳐서, 반투명)
    fig = px.histogram(data_frame=df, x=col, nbins=30, title=f'품종별 {title} 분포',
                       color=label, barmode='overlay', opacity=0.6)
    fig.show()

In [ ]:
# 꽃잎의 길이 vs 꽃잎의 너비 관계 → 품종이 거의 겹치지 않고 무리 지어 나뉜다
fig = px.scatter(data_frame=df, x=features[2], y=features[3], color=label,
                 title='꽃잎의 길이와 꽃잎의 너비 관계')
fig.show()

# 숫자로 확인 : 품종별 꽃잎 크기 범위
print(df.groupby(label)[[features[2], features[3]]].agg(['min', 'max']))

> **EDA 결과** — 꽃받침(sepal)은 품종끼리 많이 겹치지만, **꽃잎(petal)** 은 품종별로 뚜렷이 나뉜다.
> 특히 setosa 는 꽃잎 길이 1.0~1.9, 너비 0.1~0.6 으로 나머지와 **완전히 떨어져** 있다 → 트리의 첫 질문이 꽃잎이 될 거라고 예상할 수 있다.

> **📝 고친 부분** (`붓꽃데이터EDA`)
> - 특성 4개마다 똑같은 히스토그램 코드 2개씩(총 8셀)을 복사해 쓰던 것 → **`for` 반복문 한 셀**로
> - 제목 "꽃잎**이** 길이와 꽃잎의 너비 관계" → 꽃잎**의**
> - 산점도에서 본 "품종이 나뉜다" 를 숫자(`groupby` 최솟값 · 최댓값)로 확인하는 코드 추가
> - 경로 `C:\Users\Jinho\work\iris_dataset.csv` → `./iris_dataset.csv`, `warnings.filterwarnings('ignore')` 제거, `display(df)` → 모양 출력

## 5. 붓꽃 품종 분류 — `DecisionTreeClassifier` (교재 p.79~81)

```python
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(매개변수)         # 모델 생성
dt.fit(X_train, y_train)                      # 학습
pred_test = dt.predict(X_test)                # 예측
acc_dt = accuracy_score(y_test, pred_test)    # 평가
```

### 5-1. 레이블 인코딩 · 데이터 분할

In [ ]:
# 품종(문자열) → 숫자 : 레이블 인코딩 (Day 020)
#   replace 는 자료형이 object 로 남을 수 있어서, 컬럼 하나만 바꿀 때는 map 이 깔끔하다
label_map = {'setosa': 0, 'versicolor': 1, 'virginica': 2}

df_label = df.copy()
df_label['label'] = df['label'].map(label_map)
print(df_label['label'].value_counts().sort_index())

In [ ]:
from sklearn.model_selection import train_test_split

X_data = df_label.drop(columns='label')    # 특성 4개 (문제)
y_data = df_label['label']                 # 정답 (품종 번호)

X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=0,
    stratify=y_data      # 품종 비율 1:1:1 유지
)

print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)
print(f"학습용 정답별 빈도수 : {y_train.value_counts().sort_index().tolist()}")

> 트리는 **표준화를 하지 않는다.** 한 특성 안에서 "몇 이하인가" 로만 자르기 때문에 단위나 범위가 결과에 영향을 주지 않는다.

### 5-2. 학습 · 트리 시각화

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# 모델 생성 → random_state 를 고정해야 매번 같은 트리가 나온다 (5-4 참고)
dt = DecisionTreeClassifier(random_state=0)

# 학습 : 학습용 특성(X_train)과 정답(y_train)으로 분할 규칙 찾기
dt.fit(X_train, y_train)
print("학습 완료! 깊이 :", dt.get_depth(), "/ 리프 노드 수 :", dt.get_n_leaves())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 8))     # 그림 크기 (가로, 세로 인치)

plot_tree(
    decision_tree=dt,
    feature_names=X_train.columns,                    # 분할 조건에 특성 이름 표시
    class_names=['setosa', 'versicolor', 'virginica'],# 리프에 품종 이름 표시
    filled=True,                                      # 다수 클래스 색으로 상자 칠하기
    fontsize=9
)
plt.show()

> **트리 읽기** (교재 p.70 과 같은 구조)
> - 루트 `petal width (cm) <= 0.8` · `gini = 0.667` · `samples = 120` · `value = [40, 40, 40]`
> - 참(왼쪽) → `gini = 0.0` · `value = [40, 0, 0]` → **setosa 40개가 한 번에 완벽하게 분리**
> - 거짓(오른쪽) → `petal width <= 1.75` 로 versicolor / virginica 를 다시 나눈다

### 5-3. 예측 · 평가 · 클래스 확률

In [ ]:
from sklearn.metrics import accuracy_score

# 평가용 데이터로 예측
pred_test = dt.predict(X_test)
print("예측 :", pred_test)
print("정답 :", y_test.values)

# 정확도
acc_dt = accuracy_score(y_test, pred_test)
print(f"정확도 : {acc_dt:.4f}  ({len(y_test)}개 중 {(pred_test == y_test).sum()}개)")

In [ ]:
# 특성 중요도 : 분할로 불순도를 얼마나 줄였는가 (합계 1)
importance = pd.Series(dt.feature_importances_, index=X_data.columns)
print(importance.round(3).sort_values(ascending=False))

In [ ]:
# 클래스 확률 : 도착한 리프 노드의 클래스 비율
#   끝까지 자란 트리는 리프가 순수(불순도 0)해서 확률이 0 아니면 1
print("끝까지 자란 트리 :")
print(dt.predict_proba(X_test[:3]))

print("-" * 40)

# 깊이를 2 로 제한하면 리프에 여러 품종이 섞여 확률이 중간값이 된다
dt2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print("max_depth=2 트리 :")
print(dt2.predict_proba(X_test[:4]).round(3))
print(f"max_depth=2 정확도 : {accuracy_score(y_test, dt2.predict(X_test)):.4f}")

> **결과 해석**
> - 정확도 **0.9667** — 평가용 30개 중 **29개** 정답
> - 특성 중요도는 꽃잎 너비 **0.955** — 거의 꽃잎 너비 하나로 분류하고 있다 (EDA 예상과 같음)
> - `max_depth=2` 트리의 versicolor 리프는 `[0, 39, 3]` 처럼 섞여 있어 확률이 **[0, 0.929, 0.071]** = 39/42, 3/42 로 나온다 — 교재 p.72 공식 그대로

### 5-4. `random_state` — 재현성 (교재 p.81)

같은 불순도 감소량을 내는 분할 기준이 여러 개일 수 있다. `random_state` 는 그중 **어느 것을 고를지**에 영향을 주고, 그 선택에 따라 아래 단계의 트리 모양이 달라질 수 있다.

In [ ]:
# random_state 를 바꿔 가며 루트 노드의 분할 조건 비교
for rs in [0, 42]:
    tree = DecisionTreeClassifier(random_state=rs).fit(X_train, y_train)
    root_feature = X_data.columns[tree.tree_.feature[0]]
    root_threshold = tree.tree_.threshold[0]
    acc = accuracy_score(y_test, tree.predict(X_test))
    print(f"random_state={rs:>2} → 루트 : {root_feature} <= {root_threshold:.2f} / 정확도 {acc:.4f}")

> `random_state=0` 은 **꽃잎 너비 <= 0.8**, `random_state=42` 는 **꽃잎 길이 <= 2.45** 로 루트를 나눴다 — 교재 p.81 의 두 그림과 같다.
> 둘 다 setosa 40개를 완벽하게 떼어 내는 **똑같이 좋은 분할**이라 어느 쪽이든 고를 수 있기 때문이다. 결과를 재현하려면 `random_state` 를 **고정**한다.

> **📝 고친 부분** (`결정트리분류실습` · `SVM_분류실습` 뒷부분)
> - `train_test_split(` 을 **빈 괄호**로 실행해서 `NameError` 로 끝난 셀 → 완성 (`결정트리분류실습` 에서 이어서 완성했던 코드로 정리)
> - `dt = DecisionTreeClassifier()` 를 데이터를 불러오기도 전에 만들고, 아래에서 **또 만들던** 것 → 학습 직전에 한 번만, `random_state=0` 고정
> - 주석 "#X 학습용(문제), y **평가용**(정답)" → `X_train` · `y_train` 은 둘 다 **학습용** (문제 / 정답)
> - 학습 셀의 주석 "#평가용 데이터 이용한 예측" → 이 셀은 **학습**이다
> - 오타 — `붗꽃`, `커럼`, `변호한`, `cout`
> - `df.replace({...})` → `df['label'].map({...})` 로 바꾼 필기의 선택 유지 (자료형이 처음부터 정수)
> - `plot_tree` 에 `class_names` · `filled=True` 추가 → 리프에 품종 이름과 색이 나와 읽기 쉬움
> - 교재 p.72 클래스 확률(`predict_proba`), p.81 `random_state` 비교, 특성 중요도 추가

## 6. 당뇨병 여부 분류 — Diabetes Health Indicators (교재 p.75~79)

미국 질병통제예방센터(CDC)가 매년 수집하는 건강 관련 전화 설문조사(BRFSS 2015) 데이터.
**70,692개**, 컬럼 22개 (정답 1 + 특성 21). 당뇨 / 정상이 **50 : 50** 으로 맞춰진 버전이다.

| 컬럼 | 뜻 | 값 |
|------|-----|-----|
| **`Diabetes_binary`** | **정답** | 0 정상, 1 당뇨병 + 당뇨병 전단계 |
| `HighBP` · `HighChol` | 고혈압 · 고지혈 | 0 / 1 |
| `CholCheck` | 5년 안에 콜레스테롤 검사 | 0 / 1 |
| `BMI` | 체질량지수 = 몸무게(kg) ÷ 키(m)² | 숫자 |
| `Smoker` | 평생 담배 100개비 이상 | 0 / 1 |
| `Stroke` · `HeartDiseaseorAttack` | 뇌졸중 · 심장 질환 | 0 / 1 |
| `PhysActivity` | 지난 30일 운동 | 0 / 1 |
| `Fruits` · `Veggies` | 하루 1회 이상 과일 · 채소 | 0 / 1 |
| `HvyAlcoholConsump` | 과음 (남 주 14잔 · 여 주 7잔 이상) | 0 / 1 |
| `AnyHealthcare` · `NoDocbcCost` | 건강 보험 보유 · 비용 때문에 병원 못 감 | 0 / 1 |
| `GenHlth` | 스스로 평가한 전반적 건강 | 1 매우 좋음 ~ 5 나쁨 |
| `MentHlth` · `PhysHlth` | 지난 30일 중 정신 · 신체 건강이 나빴던 날 수 | 0 ~ 30 |
| `DiffWalk` | 걷기 · 계단 오르기가 심각하게 어려움 | 0 / 1 |
| `Sex` | 성별 (교재에서는 `Gender`) | 0 여성, 1 남성 |
| `Age` | 나이 13단계 | 1 (18~24세) ~ 13 (80세 이상) |
| `Education` | 교육 수준 6단계 | 1 ~ 6 |
| `Income` | 연간 가구 소득 8단계 | 1 ($10,000 미만) ~ 8 ($75,000 이상) |

In [ ]:
# 데이터 불러오기 (노트북과 같은 폴더, 약 6MB)
path = './diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
df_db = pd.read_csv(path)

print("모양 :", df_db.shape)
print("누락 데이터 :", df_db.isnull().sum().sum(), "개")
print(f"정답 분포 : \n{df_db['Diabetes_binary'].value_counts()}")
df_db.head()

In [ ]:
# 정답별 평균 비교 → 당뇨 그룹에서 크게 달라지는 특성은?
diff = df_db.groupby('Diabetes_binary').mean().T
diff.columns = ['정상(0)', '당뇨(1)']
diff['차이'] = diff['당뇨(1)'] - diff['정상(0)']
print(diff.round(2).sort_values('차이', key=abs, ascending=False).head(8))

In [ ]:
# 특성(21개) / 정답 나누기 → 80 : 20 분할
X_data = df_db.drop(columns=['Diabetes_binary'])
y_data = df_db['Diabetes_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=0, stratify=y_data
)
print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)

In [ ]:
# 결정 트리 : 깊이 제한 없이 (끝까지 분할)
dt_full = DecisionTreeClassifier(random_state=0)
dt_full.fit(X_train, y_train)

print("깊이 :", dt_full.get_depth(), "/ 리프 노드 수 :", dt_full.get_n_leaves())
print(f"학습용 정확도 : {accuracy_score(y_train, dt_full.predict(X_train)):.4f}")
print(f"평가용 정확도 : {accuracy_score(y_test, dt_full.predict(X_test)):.4f}")

> **과대적합!** 깊이 37, 리프 1만 6천여 개까지 자라서 학습용은 **99.5%** 를 맞히지만 평가용은 **65.5%** 에 그친다.
> 학습용 데이터의 우연한 특징까지 외워 버린 것이다. → `max_depth` 로 깊이를 제한해 본다.

In [ ]:
# max_depth (최대 깊이) 를 바꿔 가며 학습용 / 평가용 정확도 비교
rows = []
for depth in [3, 5, 7, 9, 12, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({
        'max_depth': '제한 없음' if depth is None else depth,
        '리프 수': model.get_n_leaves(),
        '학습용 정확도': round(accuracy_score(y_train, model.predict(X_train)), 4),
        '평가용 정확도': round(accuracy_score(y_test, model.predict(X_test)), 4),
    })

pd.DataFrame(rows)

In [ ]:
# 깊이 7 트리 : 평가용 정확도 + 어떤 특성으로 나눴나
dt7 = DecisionTreeClassifier(max_depth=7, random_state=0).fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt7.predict(X_test))
print(f"max_depth=7 평가용 정확도 : {acc_dt:.4f}")

print("-" * 40)
importance = pd.Series(dt7.feature_importances_, index=X_data.columns)
print(importance.round(3).sort_values(ascending=False).head(6))

In [ ]:
# 위쪽 3단계만 그려 보기 (전체는 너무 크다)
plt.figure(figsize=(18, 7))
plot_tree(dt7, feature_names=X_data.columns, class_names=['normal', 'diabetes'],
          filled=True, max_depth=2, fontsize=9)
plt.show()

> **결과 해석**
> - 깊이를 **3 → 9** 로 늘리면 평가용 정확도가 **71.3% → 74.3%** 로 오르다가, 12 부터는 학습용만 오르고 평가용은 **떨어진다** — 과대적합의 시작
> - 깊이 7 정도면 학습용 74.8% · 평가용 **74.1%** 로 차이가 거의 없다
> - 중요한 특성은 **고혈압(`HighBP`) 0.45**, **전반적 건강(`GenHlth`) 0.28**, `BMI`, `Age` 순
> - 정답이 50 : 50 이라 "전부 한쪽으로 찍기" 기준선은 **50%** 다

> **📝 고친 부분** (`결정트리분류실습02`)
> - 데이터를 불러와 `display(df)` 한 뒤 **빈 셀로 끝나 있던** 것 → 교재 p.79 순서(분할 → 생성 → 학습 → 예측 → 평가)대로 완성
> - 교재 컬럼 정보(p.76~78)를 표로 정리 — 교재의 `Gender` 는 실제 파일에서 **`Sex`**
> - 교재 p.77 `PhysHlth` 답변 "**정신** 건강이 좋지 않았던 일 수" → **신체** 건강 (교재 오타)
> - `max_depth` 에 따른 과대적합 비교 추가

## 7. 오늘 정리 (결정 트리)

| 항목 | 내용 |
|------|------|
| 원리 | 특성 하나의 **임계값**으로 데이터를 **둘로** 반복 분할 → 불순도 최소화 (CART) |
| 불순도 | **지니** $1 - \sum p^2$ (기본값) · **엔트로피** $-\sum p \log_2 p$. 순수하면 0 |
| 구성 | 루트 → 결정 노드 → **리프**(최종 예측). 노드 정보 = 조건 · 불순도 · samples · value |
| 확률 | 리프에 있는 학습 데이터의 **클래스 비율** = `predict_proba` |
| 장점 | **설명 가능** (IF-THEN), **표준화 불필요** |
| 주의 | 끝까지 자라면 **과대적합** → `max_depth` 로 제한, 재현하려면 `random_state` 고정 |

| 데이터 | 결과 |
|--------|------|
| 붓꽃 (150개) | 정확도 **0.9667**, 꽃잎 너비 중요도 0.955 |
| 당뇨병 (70,692개) | 제한 없음 65.5% → `max_depth=7` **74.1%** |

- 누락 · 중복 데이터를 **이유 없이 지우면** 데이터가 한쪽으로 치우칠 수 있다 → 지우기 전에 **얼마나, 왜** 빠졌는지 먼저 확인한다. (`notbook04.py` 메모 "잘못하면 데이터 조작이 된다" 를 이 뜻으로 정리)